# Assignment 2 — Stage 2: Hidden Test Evaluation

In [ ]:
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from torch import nn

CHECKPOINT_DIR = Path("model_checkpoint")

# The instructor's released file is named hidden_test_with_labels.csv here.
# If your local filename is different, change only this path.
HIDDEN_TEST_PATH = Path("hidden_test_with_labels.csv")

hidden_df = pd.read_csv(HIDDEN_TEST_PATH)

print("Hidden-test shape:", hidden_df.shape)
print("Columns:", hidden_df.columns.tolist())
print("\nLabel counts:")
print(hidden_df["label"].value_counts().sort_index())

## Load the same Stage 1 model

In [ ]:
with open(CHECKPOINT_DIR / "config.json", encoding="utf-8") as file:
    config = json.load(file)

with open(CHECKPOINT_DIR / "vocab.json", encoding="utf-8") as file:
    vocab = json.load(file)

with open(CHECKPOINT_DIR / "metrics.json", encoding="utf-8") as file:
    stage1_metrics = json.load(file)

class SentimentEmbeddingBag(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=64, dropout=0.4):
        super().__init__()
        self.embedding = nn.EmbeddingBag(vocab_size, embedding_dim, mode="mean")
        self.network = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 2),
        )

    def forward(self, text, offsets):
        return self.network(self.embedding(text, offsets))

model = SentimentEmbeddingBag(
    config["vocab_size"],
    config["embedding_dim"],
    config["hidden_dim"],
    config["dropout"],
)

model.load_state_dict(
    torch.load(CHECKPOINT_DIR / "model_state.pt", map_location="cpu")
)
model.eval()

print("Exact Stage 1 checkpoint loaded successfully.")

In [ ]:
TOKEN_RE = re.compile(config["token_pattern"])

def tokenize(text):
    return TOKEN_RE.findall(str(text).lower())

def encode(text):
    ids = [vocab.get(word, 0) for word in tokenize(text)]
    return ids or [0]

## Hidden-test inference

The following cell only performs forward passes through the frozen Stage 1 model. It does not update any model parameters.

In [ ]:
hidden_predictions = []

with torch.no_grad():
    for text in hidden_df["text"]:
        token_ids = torch.tensor(encode(text), dtype=torch.long)
        offsets = torch.tensor([0], dtype=torch.long)

        logits = model(token_ids, offsets)
        predicted_label = int(logits.argmax(dim=1).item())
        hidden_predictions.append(predicted_label)

print("Generated predictions:", len(hidden_predictions))

## Hidden-test accuracy and confusion matrix

In [ ]:
hidden_accuracy = accuracy_score(hidden_df["label"], hidden_predictions)
hidden_confusion = confusion_matrix(hidden_df["label"], hidden_predictions)

print(f"Hidden-test accuracy: {hidden_accuracy:.4f}")
print("Confusion matrix:")
print(hidden_confusion)

ConfusionMatrixDisplay(
    confusion_matrix=hidden_confusion,
    display_labels=["Negative (0)", "Positive (1)"],
).plot()
plt.title("Stage 2 Hidden-Test Confusion Matrix")
plt.show()

### Hidden-test result

The exact Stage 1 checkpoint achieved **0.6433 (64.33%)** accuracy on the hidden test set.

The confusion matrix is:

```text
[[165 135]
 [ 79 221]]
```

With true labels as rows and predicted labels as columns:

- **True negatives:** 165
- **False positives:** 135
- **False negatives:** 79
- **True positives:** 221

## Save `hidden_test_predictions.csv`

The assignment requires exactly two columns: `id` and `predicted_label`.

In [ ]:
predictions_df = pd.DataFrame({
    "id": hidden_df["id"],
    "predicted_label": hidden_predictions,
})

predictions_df.to_csv("hidden_test_predictions.csv", index=False)

display(predictions_df.head())
print("Saved hidden_test_predictions.csv")

## Public-test vs. hidden-test comparison

The Stage 1 public-test accuracy was **0.6675 (66.75%)**.

The Stage 2 hidden-test accuracy was **0.6433 (64.33%)**.

The hidden-test accuracy was **2.42% lower** than the public-test accuracy. The results are close, which suggests that the model generalized to unseen movie reviews, although its overall accuracy remains moderate.

## Use of AI

Generative AI was used to help organize the Stage 2 notebook, run the already-submitted model checkpoint, and explain the evaluation results. No Stage 2 data was used to retrain or modify the Stage 1 model.